# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [1]:
%pip install -q feedparser trafilatura google-generativeai python-dotenv edge-tts nest_asyncio openai-whisper moviepy==1.0.3 Pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install python-dateutil

## Paso 1: Obtener noticias

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=1, periodo='dia')

RECOPILANDO NOTICIAS — último/a dia

-> Leyendo RSS de todas las fuentes...
  [ABC] 1 articulos en RSS
  [ElDiario] 1 articulos en RSS
  [Europa Press] 1 articulos en RSS
  [La Vanguardia] 1 articulos en RSS
  inutos] 1 articulos en RSS
  [Antena 3] 1 articulos en RSS
  [RTVE] 1 articulos en RSS
  [El Pais] 1 articulos en RSS
  [El Mundo] 1 articulos en RSS

-> 9/9 artículos dentro del período 'dia'

-> Extrayendo texto completo (9 articulos)...
  Procesando: La asesora de Begoña Gómez alega que ha dedicado «medio...
    -> Texto completo (4531 chars)
  Procesando: Sánchez pide "no doblegarse" ante Estados Unidos para c...
    -> Texto completo (2093 chars)
  Procesando: El PP adelanta que no se opondrá al decreto ley de medi...
    -> Texto completo (1413 chars)
  Procesando: Primeras señales de alarma ante la falta de suministro ...
    -> Texto completo (5533 chars)
  Procesando: El PP se abstendrá en la votación del decreto de ayudas...
    -> Texto completo (27865 chars)
  Procesa

In [2]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ABC
Título:  La asesora de Begoña Gómez alega que ha dedicado «medio minuto» al día a las actividades profesionales de la mujer de Sánchez
Origen:  completo
Chars:   4531
Preview: La asesora de Begoña Gómez alega que ha dedicado «medio minuto» al día a las actividades profesionales de la mujer de Sánchez
La trabajadora de Moncloa Cristina Álvarez cuestiona el delito de malversa...

Artículo 2
Fuente:  ElDiario
Título:  Sánchez pide "no doblegarse" ante Estados Unidos para construir una Europa "más autónoma y más libre"
Origen:  completo
Chars:   2093
Preview: Sánchez pide “no doblegarse” ante Estados Unidos para construir una Europa “más autónoma y más libre”
13
Pedro Sánchez llama a Europa a “no doblegarse” ante los Estados Unidos de Donald Trump para con...

Artículo 3
Fuente:  Europa Press
Título:  El PP adelanta que no se opondrá al decreto ley de medidas fiscales por Irán y garantiza su aprobación en el Congreso
Origen:  completo
Chars:   1413
Preview: El pres

## Paso 2: Generación del resumen

In [3]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
#GEMINI_API_KEY = ""

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 

C:\Users\flore\AppData\Local\Temp\ipykernel_18648\1901222917.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai



RESUMIENDO 9 ARTÍCULOS

-> Agrupando artículos por temas...
   6 temas identificados:
   - Investigación sobre Begoña Gómez (1 fuentes: ABC)
   - Postura de Sánchez sobre autonomía europea (1 fuentes: ElDiario)
   - Aprobación de medidas económicas y decretos en el Congreso (3 fuentes: Europa Press, 20minutos, RTVE)
   - Crisis de suministro energético (1 fuentes: La Vanguardia)
   - Caso de eutanasia (1 fuentes: Antena 3)
   - Escándalo de abusos en la Iglesia (1 fuentes: El Pais)

-> Generando resúmenes por tema...
   [1/6] Investigación sobre Begoña Gómez (1 fuentes)...
      Límite de cuota, esperando 60s (intento 1/3)...


AttributeError: type object 'datetime.time' has no attribute 'sleep'

## Paso 3: Síntesis de voz (T2S)

In [6]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       3589
  Audio:       audios/audio_2026-03-26_12-11-12.mp3
Audio generado! (1.26 MB)
  Generando subtítulos con Whisper...


c:\Users\fcalv\anaconda3\envs\MCD_25_26\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Nombres corregidos en subtítulos
  Subtítulos generados: audios/subtitulos_2026-03-26_12-11-12.vtt


In [7]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 5416 chars
WEBVTT

00:00:00.000 --> 00:00:05.459
economía y energía. Bienvenidos a
al DIA. La actual situación

00:00:05.459 --> 00:00:08.880
en oriente próximo está generando
diversas repercusiones en el ámbito

00:00:08.880 --> 00:00:13.380
económico y energético global. La
vanguardia en forma de una

00:00:


In [8]:
# Paso 3b — Regenerar subtítulos con el nuevo formato (2 líneas)
import importlib
import t2s
importlib.reload(t2s)

from t2s import generar_subtitulos_whisper

ruta_subtitulos = generar_subtitulos_whisper(
    ruta_audio,
    ruta_audio.replace("audio_", "subtitulos_").replace(".mp3", ".vtt")
)


  Generando subtítulos con Whisper...


c:\Users\fcalv\anaconda3\envs\MCD_25_26\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Nombres corregidos en subtítulos
  Subtítulos generados: audios/subtitulos_2026-03-26_12-11-12.vtt


## Paso 4: Generación del video resumen 

In [ ]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')
#PEXELS_API_KEY = ""

ruta_video = generar_video(
    ruta_audio, ruta_subtitulos, resumenes,
    PEXELS_API_KEY,
    modelo_ia=mi_modelo_gemini  # para mejorar las queries de imagen
)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    220.3s
   Temas:             4
   Duración por tema: 55.1s

2. Buscando imágenes en Pexels...
  Tema: Economía y Energía...
    Query imagen: 'Energy Crisis'
    Keywords: 'energy crisis'
    Imagen guardada: imagenes/img_2037152108844393177.jpg
  Tema: Política Española...
    Query imagen: 'Election Spain'
    Keywords: 'election spain'
    Imagen guardada: imagenes/img_1654720580210493370.jpg
  Tema: Conflicto Internacional...
    Query imagen: 'Geopolitical Crisis'
    Keywords: 'geopolitical crisis'
    Imagen guardada: imagenes/img_265674555827713900.jpg
  Tema: Abusos y Ética Social...
    Query imagen: 'Rights Struggle'
    Keywords: 'rights struggle'
    Imagen guardada: imagenes/img_8770735124047336390.jpg

3. Creando clips...
   [1/4] Economía y Energía...
   [2/4] Política Española...
   [3/4] Conflicto Internacional...
   [4/4] Abusos y Ética Social...

4. Concatenando clips y añadiendo audio...

5. Exportando ví